# Regression Analysis: Oil Price Shocks and Sovereign CDS Spreads

This notebook estimates panel regressions and threshold regressions to test
whether oil-exporting sovereigns exhibit **differential sensitivity** of CDS spread
movements to oil price shocks, relative to a control group of non-oil-exporting
emerging markets.

**Core identification strategy.** We interact Brent crude log-returns
($r_t^{\text{Brent}}$) with an oil-exporter indicator ($D_i^{\text{oil}}$) and test
whether the interaction coefficient is negative and significant — i.e., whether oil
exporters' CDS spreads widen *more* when oil prices fall than those of the control
group.

---

## 0. Imports

In [14]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import datetime
from linearmodels.panel import PanelOLS

## 1. Build the dataset

We load seven data sources and merge them on the weekly date index:

| Source | Contents |
|--------|----------|
| `Weekly_CDS.csv` | Sovereign 5-year CDS spreads (wide format, one column per country) |
| `oil_prices_datastream.csv` | Brent crude prices |
| `macro_risk_variables.csv` | DXY, US Treasury yields (2Y, 5Y, 10Y) |
| `VIXCLS.csv` | CBOE VIX index |
| `OVXCLS.csv` | CBOE Oil Volatility Index (OVX) |
| `Daily_FX_Rates.csv` | USD/local-currency exchange rates (wide, one column per country) |
| `mscicountryindex.csv` | MSCI country equity indices (wide, one column per country) |

CDS, oil, macro, VIX, and OVX are **global** variables (same value for every country
in a given week). FX rates and MSCI indices are **country-specific** and are joined
during the panel reshape step.

The sample starts on **2014-01-01** onward (post-OVX availability and coinciding with
the 2014 oil price collapse, which provides a natural starting point for studying
oil-sovereign transmission).

In [15]:
# ── Load individual datasets ────────────────────────────────────────────
CDS_data = pd.read_csv('../data/processed/CDS/Weekly_CDS.csv', parse_dates=['date'], index_col='date')
Oil_data = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'], index_col='date')
Macro    = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv', parse_dates=['Date'], index_col='Date')
VIX      = pd.read_csv('../data/processed/Macroeconomic_variables/VIXCLS.csv', parse_dates=['Date'], index_col='Date')
OVX      = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv', parse_dates=['date'], index_col='date')
# Country-specific data
FX_data   = pd.read_csv('../data/processed/Macroeconomic_variables/Daily_FX_Rates.csv', parse_dates=['Date'], index_col='Date')
MSCI_data = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['date'], index_col='date')

# ── Merge global variables on date index ────────────────────────────────
merged = (
    CDS_data
    .join(Oil_data, how='inner')
    .join(Macro,    how='inner')
    .join(VIX,      how='inner')
    .join(OVX,      how='inner')
)

# Filter to 2014+
merged = merged.loc[(merged.index >= '2014-01-01') & (merged.index <= '2024-12-31')].copy()

print(f"Merged shape  : {merged.shape}")
print(f"Date range    : {merged.index.min().date()} -> {merged.index.max().date()}")
print(f"FX countries  : {FX_data.shape[1]}")
print(f"MSCI countries: {MSCI_data.shape[1]}")

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_75009/2966486902.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Oil_data = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'], index_col='date')
/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_75009/2966486902.py:4: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  Macro    = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv', parse_dates=['Date'], index_col='Date')


Merged shape  : (574, 83)
Date range    : 2014-01-03 -> 2024-12-27
FX countries  : 86
MSCI countries: 84


/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_75009/2966486902.py:9: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  MSCI_data = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['date'], index_col='date')


## 2. Variable construction

All *price-level* variables are transformed to **log-returns**:

$$
r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)
$$

This applies to: Brent, OVX, VIX, DXY, CDS spreads, FX rates, and MSCI indices.

The exception is US Treasury yields, which are already in percentage-point units;
for these we take the **first difference** ($\Delta y_t = y_t - y_{t-1}$), which is
standard in the fixed-income literature.

### Country groups

| Group | Countries |
|-------|-----------|
| **Oil exporters** | Saudi Arabia, Abu Dhabi, Dubai, Qatar, Colombia, Mexico, Brazil, Egypt, Malaysia |
| **Controls** | Indonesia, Philippines, Turkey, Chile, China, South Africa, South Korea, Thailand |

We reshape from wide to a balanced **panel** (long format) with one row per
country-week. Country-specific variables (CDS, FX, MSCI) are matched by country
name during the reshape. We then compute log-returns within each country group
and create interaction terms.

In [16]:
# ── Country groups ──────────────────────────────────────────────────────
oil_exporters = [
    'Saudi Arabia', 'Abu Dhabi', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia'
]
controls = [
    'Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand'
]

# ── Compute log-returns for global price-level variables ───────────────
merged['Oil_ret'] = np.log(merged['Brent']   / merged['Brent'].shift(1))
merged['OVX_chg'] = np.log(merged['OVXCLS']  / merged['OVXCLS'].shift(1))
merged['VIX_chg'] = np.log(merged['VIXCLS']  / merged['VIXCLS'].shift(1))
merged['DXY_chg'] = np.log(merged['DXY']     / merged['DXY'].shift(1))

# First differences for Treasury yields (basis-point changes)
merged['UST2Y_chg']  = merged['UST2Y'].diff()
merged['UST5Y_chg']  = merged['UST5Y'].diff()
merged['UST10Y_chg'] = merged['UST10Y'].diff()

# ── Reshape to long (panel) format ─────────────────────────────────────
global_vars = [
    'Brent', 'Oil_ret', 'OVXCLS', 'OVX_chg',
    'VIXCLS', 'VIX_chg', 'DXY_chg',
    'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg'
]

merged = merged.reset_index()          # bring Date back as column

# Align FX and MSCI to the same date index as merged
fx_aligned   = FX_data.reindex(pd.to_datetime(merged['date'])).reset_index()
msci_aligned = MSCI_data.reindex(pd.to_datetime(merged['date'])).reset_index()

panel_rows = []
for country in oil_exporters + controls:
    if country not in merged.columns:
        print(f"Warning: {country} not in CDS data -- skipping")
        continue

    temp = merged[['date'] + global_vars + [country]].copy()
    temp = temp.rename(columns={country: 'CDS'})
    temp['Country']     = country
    temp['OilExporter'] = int(country in oil_exporters)

    # ── Match country-specific FX rate ─────────────────────────────────
    if country in fx_aligned.columns:
        temp['FX'] = fx_aligned[country].values
    else:
        print(f"  FX missing for {country} -- filling NaN")
        temp['FX'] = np.nan

    # ── Match country-specific MSCI index ──────────────────────────────
    if country in msci_aligned.columns:
        temp['MSCI'] = msci_aligned[country].values
    else:
        print(f"  MSCI missing for {country} -- filling NaN")
        temp['MSCI'] = np.nan

    panel_rows.append(temp)

panel = (
    pd.concat(panel_rows, ignore_index=True)
    .sort_values(['Country', 'date'])
    .reset_index(drop=True)
)

# ── Country-specific log-returns (groupby to avoid cross-country diffs) ─
panel['CDS_ret'] = panel.groupby('Country')['CDS'].transform(
    lambda s: np.log(s / s.shift(1))
)
panel['FX_ret'] = panel.groupby('Country')['FX'].transform(
    lambda s: np.log(s / s.shift(1))
)
panel['MSCI_ret'] = panel.groupby('Country')['MSCI'].transform(
    lambda s: np.log(s / s.shift(1))
)

# ── Interaction terms ─────────────────────────────────────────────────
panel['OVX_x_Exporter']    = panel['OVX_chg'] * panel['OilExporter']
panel['Brent_x_Exporter']  = panel['Oil_ret'] * panel['OilExporter']

# ── Drop incomplete rows ──────────────────────────────────────────────
reg_vars = ['CDS_ret', 'OVX_chg', 'VIX_chg', 'DXY_chg',
            'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg']
panel = panel.dropna(subset=reg_vars).reset_index(drop=True)

# ── Report coverage ───────────────────────────────────────────────────
print(f"Panel shape  : {panel.shape}")
print(f"Countries    : {panel['Country'].nunique()}")
print(f"Weeks/country: ~{len(panel) // panel['Country'].nunique()}")

# Check FX/MSCI coverage
fx_coverage   = panel.groupby('Country')['FX_ret'].apply(lambda s: s.notna().mean())
msci_coverage = panel.groupby('Country')['MSCI_ret'].apply(lambda s: s.notna().mean())
coverage = pd.DataFrame({'FX_coverage': fx_coverage, 'MSCI_coverage': msci_coverage})
print("\nCountry-specific variable coverage (fraction non-NaN):")
print(coverage.to_string())
panel.head()

Panel shape  : (8592, 21)
Countries    : 16
Weeks/country: ~537

Country-specific variable coverage (fraction non-NaN):
              FX_coverage  MSCI_coverage
Country                                 
Abu Dhabi             1.0            1.0
Brazil                1.0            1.0
Chile                 1.0            1.0
China                 1.0            1.0
Colombia              1.0            1.0
Egypt                 1.0            1.0
Indonesia             1.0            1.0
Malaysia              1.0            1.0
Mexico                1.0            1.0
Philippines           1.0            1.0
Qatar                 1.0            1.0
Saudi Arabia          1.0            1.0
South Africa          1.0            1.0
South Korea           1.0            1.0
Thailand              1.0            1.0
Turkey                1.0            1.0


,date,Brent,Oil_ret,OVXCLS,OVX_chg,VIXCLS,VIX_chg,DXY_chg,UST2Y_chg,UST5Y_chg,...,CDS,Country,OilExporter,FX,MSCI,CDS_ret,FX_ret,MSCI_ret,OVX_x_Exporter,Brent_x_Exporter
0,2014-01-10,106.33,-0.006935,19.47,-0.056416,12.14,-0.125260,-0.001610,-0.018,-0.103,...,55.85999,Abu Dhabi,1,3.6720,769.313,-0.008201,0.000000,0.002561,-0.056416,-0.006935
1,2014-01-17,106.89,0.005253,17.12,-0.128627,12.44,0.024411,0.007042,0.001,0.005,...,55.32999,Abu Dhabi,1,3.6720,801.257,-0.009533,0.000000,0.040684,-0.128627,0.005253
2,2014-01-24,107.45,0.005225,19.46,0.128114,18.14,0.377202,-0.009524,-0.031,-0.064,...,55.34999,Abu Dhabi,1,3.6729,828.858,0.000361,0.000245,0.033867,0.128114,0.005225
3,2014-01-31,107.13,-0.002983,20.35,0.044720,18.41,0.014775,0.010509,-0.008,-0.056,...,56.32999,Abu Dhabi,1,3.6728,822.438,0.017551,-0.000027,-0.007776,0.044720,-0.002983
4,2014-02-07,108.21,0.010031,19.03,-0.067064,15.29,-0.185695,-0.007654,-0.033,-0.048,...,56.32999,Abu Dhabi,1,3.6720,839.812,0.000000,-0.000218,0.020905,-0.067064,0.010031


## 3. Panel regressions

We estimate four nested specifications of the following general model:

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i + \gamma_t
+ \beta_1 \, r_t^{\text{Brent}}
+ \beta_2 \,\bigl(r_t^{\text{Brent}} \times D_i^{\text{oil}}\bigr)
+ \boldsymbol{\delta}'\,\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

where:
- $\Delta \ln \text{CDS}_{i,t}$ is the weekly log-return of the 5-year CDS spread for
  country $i$,
- $r_t^{\text{Brent}} = \Delta \ln \text{Brent}_t$ is the weekly Brent crude log-return,
- $D_i^{\text{oil}}$ is a time-invariant dummy equal to 1 for oil exporters,
- $\mathbf{X}_{i,t}$ includes global controls
  ($\Delta \ln \text{VIX}_t$, $\Delta \ln \text{DXY}_t$, $\Delta \text{UST10Y}_t$)
  and country-specific controls ($r_{i,t}^{\text{MSCI}}$, $r_{i,t}^{\text{FX}}$),
- $\alpha_i$ are country fixed effects, $\gamma_t$ are time fixed effects.

The **coefficient of interest** is $\beta_2$: a negative and significant estimate
implies that oil exporters' CDS spreads widen *disproportionately* when oil prices
fall (since $r^{\text{Brent}} < 0 \Rightarrow \Delta\ln\text{CDS} > 0$ for exporters
more than for controls).

| Model | Fixed effects | Controls | Notes |
|-------|--------------|----------|-------|
| 1 | None | No | Baseline: Brent return + interaction only |
| 2 | None | Yes | Adds VIX, DXY, UST10Y, MSCI, FX |
| 3 | Country | Yes | Absorbs time-invariant country heterogeneity |
| 4 | Country + Time | — | Time FE absorb all global regressors; only the interaction survives |

All models use **standard errors clustered by country**.

### 3.0 Prepare panel index

In [18]:
# linearmodels requires a MultiIndex (entity, time)
reg_panel = panel.copy()
reg_panel['Date'] = pd.to_datetime(reg_panel['date'])
reg_panel = reg_panel.set_index(['Country', 'date'])

y = reg_panel['CDS_ret']

### 3.1 Model 1 — Baseline (no fixed effects)

$$
\Delta \ln \text{CDS}_{i,t}
= \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \varepsilon_{i,t}
$$

In [19]:
X_1 = reg_panel[['Oil_ret', 'Brent_x_Exporter']]

model_1 = PanelOLS(y, X_1, entity_effects=False, time_effects=False)
results_1 = model_1.fit(cov_type='clustered', cluster_entity=True)
print(results_1.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.0740
Estimator:                   PanelOLS   R-squared (Between):             -0.0754
No. Observations:                8592   R-squared (Within):               0.0740
Date:                Sat, Apr 11 2026   R-squared (Overall):              0.0740
Time:                        01:09:18   Log-likelihood                  1.05e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      343.12
Entities:                          16   P-value                           0.0000
Avg Obs:                       537.00   Distribution:                  F(2,8590)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             81.608
                            

### 3.2 Model 2 — Baseline + global and country-specific controls (no fixed effects)

$$
\Delta \ln \text{CDS}_{i,t}
= \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \beta_3\,\Delta \ln\text{VIX}_t
+ \beta_4\,\Delta \ln\text{DXY}_t
+ \beta_5\, r_{i,t}^{\text{MSCI}}
+ \beta_6\, r_{i,t}^{\text{FX}}
+ \beta_7\,\Delta\text{UST10Y}_t
+ \varepsilon_{i,t}
$$

In [20]:
X_2 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret','FX_ret', 'UST10Y_chg']]

model_2 = PanelOLS(y, X_2, entity_effects=False, time_effects=False)
results_2 = model_2.fit(cov_type='clustered', cluster_entity=True)
print(results_2.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.3536
Estimator:                   PanelOLS   R-squared (Between):             -3.2512
No. Observations:                8592   R-squared (Within):               0.3542
Date:                Sat, Apr 11 2026   R-squared (Overall):              0.3536
Time:                        01:09:26   Log-likelihood                 1.205e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      670.98
Entities:                          16   P-value                           0.0000
Avg Obs:                       537.00   Distribution:                  F(7,8585)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             155.14
                            

### 3.3 Model 3 — Country fixed effects + controls

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i
+ \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \boldsymbol{\delta}'\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

Country fixed effects $\alpha_i$ absorb any time-invariant heterogeneity across
sovereigns (e.g., average credit quality, economic structure).

In [21]:
X_3 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret','FX_ret', 'UST10Y_chg']]

model_3 = PanelOLS(y, X_3, entity_effects=True, time_effects=False)
results_3 = model_3.fit(cov_type='clustered', cluster_entity=True)
print(results_3.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.3542
Estimator:                   PanelOLS   R-squared (Between):             -3.2738
No. Observations:                8592   R-squared (Within):               0.3542
Date:                Sat, Apr 11 2026   R-squared (Overall):              0.3536
Time:                        01:09:36   Log-likelihood                 1.205e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      671.40
Entities:                          16   P-value                           0.0000
Avg Obs:                       537.00   Distribution:                  F(7,8569)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             152.79
                            

### 3.4 Model 4 — Country + time fixed effects

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i + \gamma_t
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \varepsilon_{i,t}
$$

Time fixed effects $\gamma_t$ absorb **all** week-specific global shocks (oil price
level, VIX, DXY, Treasuries). The only regressor with cross-sectional variation that
survives is the **interaction term** — making this the cleanest test of differential
sensitivity.

In [22]:
X_4 = reg_panel[['Brent_x_Exporter']]

model_4 = PanelOLS(y, X_4, entity_effects=True, time_effects=True)
results_4 = model_4.fit(cov_type='clustered', cluster_entity=True)
print(results_4.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.0017
Estimator:                   PanelOLS   R-squared (Between):             -0.0042
No. Observations:                8592   R-squared (Within):               0.0137
Date:                Sat, Apr 11 2026   R-squared (Overall):              0.0137
Time:                        01:09:39   Log-likelihood                 1.427e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      13.374
Entities:                          16   P-value                           0.0003
Avg Obs:                       537.00   Distribution:                  F(1,8039)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             1.1998
                            

### 3.5 Comparison across specifications

We summarise the interaction coefficient $\hat{\beta}_2$ (`Brent_x_Exporter`) across
the four models to assess stability.

In [23]:
print(f"\n{'Model':<45} {'β':>10} {'SE':>10} {'p-value':>10} {'R²':>10}")
print("-" * 85)

models = [
    ('Model 1: Baseline',              results_1),
    ('Model 2: + Global Controls',     results_2),
    ('Model 3: + Country FE',          results_3),
    ('Model 4: + Country FE + Time FE', results_4),
]

for name, res in models:
    beta = res.params['Brent_x_Exporter']
    se   = res.std_errors['Brent_x_Exporter']
    pval = res.pvalues['Brent_x_Exporter']
    r2   = res.rsquared_within if hasattr(res, 'rsquared_within') else res.rsquared

    sig = ''
    if pval < 0.1:  sig = '*'
    if pval < 0.05: sig = '**'
    if pval < 0.01: sig = '***'

    print(f"{name:<45} {beta:>10.4f} {se:>10.4f} {pval:>10.4f} {r2:>10.4f} {sig}")

print("\n* p<0.1, ** p<0.05, *** p<0.01")
print("Standard errors clustered by country.")


Model                                                  β         SE    p-value         R²
-------------------------------------------------------------------------------------
Model 1: Baseline                                -0.0713     0.0630     0.2575     0.0740 
Model 2: + Global Controls                       -0.0439     0.0458     0.3381     0.3542 
Model 3: + Country FE                            -0.0439     0.0458     0.3383     0.3542 
Model 4: + Country FE + Time FE                  -0.0713     0.0651     0.2734     0.0137 

* p<0.1, ** p<0.05, *** p<0.01
Standard errors clustered by country.


## 4. Threshold regressions

To capture this nonlinearity we condition on the **OVX level** rather than OVX returns:
a high OVX level signals elevated oil-market uncertainty regardless of the weekly
change. We restrict the sample to weeks when the OVX level exceeds a given quantile
and re-estimate the panel model with Brent returns as the main regressor.

For each threshold $q \in \{80\%, 90\%, 95\%, 99\%\}$, we define:

$$
\mathcal{T}_q = \bigl\{ t : \text{OVX}_t > Q_q(\text{OVX}) \bigr\}
$$

and estimate on the restricted sample $\{(i,t) : t \in \mathcal{T}_q\}$:

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i
+ \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \boldsymbol{\delta}'\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

If the oil-exporter differential is driven by **tail events**, we should see
$|\hat{\beta}_2|$ increase in magnitude as the threshold becomes more extreme.

In [29]:
results_list = []

ovx_quantiles = {
    'Full Sample': None,
    'OVX > Q70':   panel['OVXCLS'].quantile(0.70),
    'OVX > Q80':   panel['OVXCLS'].quantile(0.80),
    'OVX > Q90':   panel['OVXCLS'].quantile(0.90),
    'OVX > Q95':   panel['OVXCLS'].quantile(0.95),
    'OVX > Q99':   panel['OVXCLS'].quantile(0.99),
}

for label, cutoff in ovx_quantiles.items():

    if cutoff is None:
        subset = panel.copy()
    else:
        subset = panel[panel['OVXCLS'] > cutoff].copy()

    subset['Date'] = pd.to_datetime(subset['date'])
    subset = subset.set_index(['Country', 'Date'])

    y_sub = subset['CDS_ret']

    # ── Spec A: OVX as regressor ──────────────────────────────────────
    X = subset[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret','FX_ret', 'UST10Y_chg']]
    res = PanelOLS(y_sub, X,
                     entity_effects=True, time_effects=False,
                     drop_absorbed=True, check_rank=False
            ).fit(cov_type='clustered', cluster_entity=True)

    results_list.append({
        'Threshold': label,
        'Cutoff':    cutoff,
        'N':         len(y_sub),
        # Brent interaction
        'β_Brent_x_exporter':   res.params['Brent_x_Exporter'],
        'SE_Brent_x_exporter':  res.std_errors['Brent_x_Exporter'],
        'p_Brent_x_exporter':   res.pvalues['Brent_x_Exporter'],
    })

# ── Print comparison ──────────────────────────────────────────────────
print(f"{'Threshold':<15} {'N':>6} │ "
      f"{'β_Brent':>8} {'p':>7}")
print("-" * 65)

for row in results_list:
    sig_b = '***' if row['p_Brent_x_exporter'] < 0.01 else '**' if row['p_Brent_x_exporter'] < 0.05 else '*' if row['p_Brent_x_exporter'] < 0.1 else ''

    print(f"{row['Threshold']:<15} {row['N']:>6} │ "
          f"{row['β_Brent_x_exporter']:>8.4f} {row['p_Brent_x_exporter']:>6.3f}{sig_b:<3}")

print("\nBoth specs: Country FE + global controls, clustered SEs by country")
print("Threshold: weeks where OVX level > quantile")

Threshold            N │  β_Brent       p
-----------------------------------------------------------------
Full Sample       8592 │  -0.0439  0.338   
OVX > Q70         2560 │  -0.0684  0.215   
OVX > Q80         1712 │  -0.1138  0.045** 
OVX > Q90          848 │  -0.1648  0.005***
OVX > Q95          416 │  -0.2366  0.000***
OVX > Q99           80 │  -0.3233  0.004***

Both specs: Country FE + global controls, clustered SEs by country
Threshold: weeks where OVX level > quantile
